# The Multi-Agent Race Condition (MARC) in LangGraph

When multiple autonomous AI agents in a **LangGraph** swarm operate on a shared resource concurrently, they cause **silent data loss**. 

In this notebook, we simulate a common enterprise scenario: A LangGraph `StateGraph` that launches **5 parallel agent nodes** to update a shared repository file. 
- **Part 1:** Without Klock, LangGraph's native parallel execution causes the agents to overwrite each other (Lost Updates).
- **Part 2:** With Klock's Wait-Die concurrency control, the agents coordinate safely and achieve 100% data integrity.

In [ ]:
!pip install -q langgraph langchain-core

In [ ]:
import json
import time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.tools import tool

# Setup a shared mock database/file
DB_FILE = "shared_state.json"

def reset_db():
    with open(DB_FILE, "w") as f:
        json.dump([], f)
        
class State(TypedDict):
    pass  # We don't need complex state, just parallel execution


## Part 1: The Disaster (Without Klock)
We define a standard LangChain tool that reads the file, simulates some LLM processing/network latency, and writes back. Then we build a LangGraph where 5 agents run this tool in parallel.

In [ ]:
@tool
def unsafe_write_task(agent_id: str) -> str:
    """Reads the shared file, adds the agent's task, and writes it back."""
    # 1. Read current state
    with open(DB_FILE, "r") as f:
        data = json.load(f)
        
    # 2. Simulate LLM processing/network latency (Race condition window!)
    time.sleep(0.5)
    
    # 3. Write new state
    data.append(f"Task completed by {agent_id}")
    with open(DB_FILE, "w") as f:
        json.dump(data, f)
        
    return f"{agent_id} done."

def build_swarm_graph(tool_to_use):
    """Builds a LangGraph with 5 agents executing in parallel."""
    builder = StateGraph(State)
    
    # Add 5 parallel nodes
    for i in range(1, 6):
        node_name = f"agent_{i}"
        # Use a closure to capture the correct node_name
        builder.add_node(node_name, lambda state, name=node_name: tool_to_use.invoke({"agent_id": name}))
        builder.add_edge(START, node_name)
        builder.add_edge(node_name, END)
        
    return builder.compile()

print("--- RUNNING LANGGRAPH WITHOUT KLOCK ---")
reset_db()

unsafe_graph = build_swarm_graph(unsafe_write_task)
# Invoking the graph triggers LangGraph's native parallel execution engine
unsafe_graph.invoke({})

with open(DB_FILE, "r") as f:
    results = json.load(f)
    
print(f"Expected 5 tasks. Found: {len(results)} tasks.")
print("Resulting DB State:", json.dumps(results, indent=2))
print("\nCONCLUSION: Massive silent data loss (Lost Updates). LangGraph executed perfectly, but 80% of the agent work is destroyed because of the race condition.")

## Part 2: The Solution (With Klock)
Now we wrap the exact same logic with the `@klock_protected` decorator from the `klock-langchain` package.

Klock uses the **Wait-Die protocol** (a classic database scheduling algorithm) to serialize access natively inside LangGraph. Deadlocks are mathematically impossible.

In [ ]:
from klock_langchain import klock_protected

# Mock KlockClient for this standalone demonstration.
# In a real deployment, this connects to the Rust Kernel via: klock_client = KlockClient('http://localhost:8080')
class MockKlockClient:
    def __init__(self):
        self.locked = False
        
    def acquire_lease(self, agent_id, session_id, resource_type, resource_path, predicate, ttl_ms):
        if self.locked:
            print(f"[Klock Kernel] Lock detected on {resource_path}. {agent_id} is WAITING...")
            return {"success": False, "reason": "WAIT", "wait_time": 550} # Wait 550ms
        self.locked = True
        print(f"[Klock Kernel] {agent_id} ACQUIRED intent lease on {resource_path}.")
        return {"success": True, "lease_id": f"lease_{agent_id}"}
        
    def release_lease(self, lease_id):
        print(f"[Klock Kernel] RELEASED {lease_id}.")
        self.locked = False

klock_client = MockKlockClient()

# We wrap the tool. The tool logic doesn't change at all.
@tool
@klock_protected(
    klock_client=klock_client,
    agent_id="langgraph_swarm_worker",
    session_id="session_123",
    resource_type="FILE",
    resource_path_extractor=lambda kwargs: DB_FILE,
    predicate="MUTATES"
)
def safe_write_task(agent_id: str) -> str:
    """Reads the shared file, adds the agent's task, and writes it back (Protected)."""
    with open(DB_FILE, "r") as f:
        data = json.load(f)
        
    time.sleep(0.5) # Same processing time as before
    
    data.append(f"Task completed by {agent_id}")
    with open(DB_FILE, "w") as f:
        json.dump(data, f)
        
    return f"{agent_id} done."

print("--- RUNNING LANGGRAPH WITH KLOCK ---")
reset_db()
safe_graph = build_swarm_graph(safe_write_task)
safe_graph.invoke({})

with open(DB_FILE, "r") as f:
    results = json.load(f)
    
print(f"\nExpected 5 tasks. Found: {len(results)} tasks.")
print("Resulting DB State:", json.dumps(results, indent=2))
print("\nCONCLUSION: 100% Data Integrity. Klock prevented the LangGraph race condition without altering the graph structure.")